# Broomhead CA - parameter panel

Change the sliders and dropdowns below, then press **Run simulation**. Results (maps, ASCII grids, summary) are written to a folder next to the code.

Requires: `pip install ipywidgets` and the Jupyter extension in VS Code (select this notebook's Python kernel).

With **Show fire-spread animation** ticked, the run also plays the fire spreading from the ignition line across the land coloured by **fire intensity** (Byram fireline intensity) in the same five classes and colours as the patch-crossing / perimeter-intensity figures: < 500 kW/m (hand tools can hold), 500-2000 (machinery needed), 2000-4000 (aircraft / indirect), 4000-10000 (indirect attack only), > 10000 (extreme), from pale yellow to dark red. The black line is the advancing fire front, blue = ignition, grey = managed patch / non-fuel, pale green = heather not reached yet; the arrow shows where the wind blows to. `intensity_map.png` shows the same classes with fire perimeters every N minutes; the animation is also saved as `fire_spread.gif` in the output folder.

Under it, the **fuel-model animation** shows the same fire, step for step, over the fuel map: heather fuel models 33-36 in green from light (shortest fuel bed) to dark (tallest), turning grey once burned (darker grey = taller fuel). Red = fire front, blue = ignition, warm grey = managed patch / non-fuel; the line under the clock gives the share of each fuel model burned so far. It is saved as `fuel_spread.gif`.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('.'))
import ipywidgets as w
from IPython.display import display, Image, clear_output
import io
from bhca.custom_run import CustomRun, run_custom, MOISTURE_PRESETS, CALIBRATED_WIND_KMH

wind_kmh = w.FloatSlider(value=22.0, min=0.0, max=30.0, step=0.5, description='Wind speed (km/h)', style={'description_width':'140px'}, layout=w.Layout(width='520px'))
wind_from = w.IntSlider(value=225, min=0, max=359, step=1, description='Wind from (deg)', style={'description_width':'140px'}, layout=w.Layout(width='520px'))
preset = w.Dropdown(options=['wet','spring','summer','drought','custom'], value='summer', description='Moisture preset', style={'description_width':'140px'})
m1 = w.FloatText(value=8, description='1-h dead (%)', style={'description_width':'140px'})
m10 = w.FloatText(value=11, description='10-h dead (%)', style={'description_width':'140px'})
m100 = w.FloatText(value=14, description='100-h dead (%)', style={'description_width':'140px'})
mherb = w.FloatText(value=60, description='Live herb (%)', style={'description_width':'140px'})
mwoody = w.FloatText(value=140, description='Live woody (%)', style={'description_width':'140px'})
temp = w.FloatSlider(value=18, min=-5, max=40, step=0.5, description='Air temp (C)', style={'description_width':'140px'})
rh = w.FloatSlider(value=63, min=5, max=100, step=1, description='Rel. humidity (%)', style={'description_width':'140px'})
ignition = w.Dropdown(options=[('East line','east'),('North line','north'),('Middle horizontal','mid_horizontal'),
                               ('Middle vertical','mid_vertical'),('Rectangle 10 m','rect10'),('Rectangle 5 m','rect5'),
                               ('West line','west'),('South line','south'),('Southwest line','southwest')], value='east', description='Ignition line', style={'description_width':'140px'})
ros_mult = w.FloatSlider(value=1.0, min=0.5, max=1.5, step=0.01, description='ROS multiplier', style={'description_width':'140px'})
engine = w.Dropdown(options=['mtt','ca'], value='mtt', description='Engine', style={'description_width':'140px'})
animate = w.Checkbox(value=True, description='Show fire-spread animation')
out_name = w.Text(value='panel_run', description='Output folder', style={'description_width':'140px'})
button = w.Button(description='Run simulation', button_style='success')
out = w.Output()

def _sync_preset(change):
    if change['new'] != 'custom':
        vals = MOISTURE_PRESETS[change['new']]
        m1.value, m10.value, m100.value, mherb.value, mwoody.value = vals
preset.observe(_sync_preset, names='value')

def fire_player(frames, fuel_frames=None, fps=10):
    """Play / pause / scrub through the fire spreading from the ignition line. With fuel_frames, the same fire is
    shown underneath on the fuel-model map, driven by the same slider."""
    def _png(fr):
        out = []
        for _, im in fr:
            b = io.BytesIO(); im.save(b, format='png'); out.append(b.getvalue())
        return out
    pngs = _png(frames)
    fuel_pngs = _png(fuel_frames) if fuel_frames else None
    view = w.Image(value=pngs[0], format='png', layout=w.Layout(width='650px'))
    fuel_view = w.Image(value=fuel_pngs[0], format='png', layout=w.Layout(width='650px')) if fuel_pngs else None
    play = w.Play(value=0, min=0, max=len(pngs) - 1, step=1, interval=int(1000 / fps))
    step = w.IntSlider(value=0, min=0, max=len(pngs) - 1, readout=False, layout=w.Layout(width='420px'))
    label = w.Label(f't = {frames[0][0]:.1f} min')
    speed = w.Dropdown(options=[('0.5x', 2 * int(1000 / fps)), ('1x', int(1000 / fps)), ('2x', int(500 / fps)), ('4x', int(250 / fps))],
                       value=int(1000 / fps), description='Speed', layout=w.Layout(width='150px'))
    w.jslink((play, 'value'), (step, 'value'))
    speed.observe(lambda ch: setattr(play, 'interval', ch['new']), names='value')
    def _show(ch):
        i = ch['new']
        view.value = pngs[i]; label.value = f't = {frames[i][0]:.1f} min'
        if fuel_view is not None:
            fuel_view.value = fuel_pngs[i]
    step.observe(_show, names='value')
    return w.VBox([w.HBox([play, step, label, speed]), view] + ([fuel_view] if fuel_view is not None else []))

def _run(_):
    with out:
        clear_output()
        p = CustomRun(wind_speed_kmh=wind_kmh.value, wind_from_deg=float(wind_from.value),
                      moisture=[m1.value, m10.value, m100.value, mherb.value, mwoody.value],
                      temperature_c=temp.value, relative_humidity_pct=rh.value, ignition=ignition.value,
                      engine=engine.value, ros_multiplier=ros_mult.value,
                      out_dir=os.path.join('..', out_name.value), make_animation=animate.value)
        print('Running...')
        res, T, ros, fl = run_custom(p)
        clear_output()
        for k, v in res.items():
            if k not in ('map', 'animation', 'fuel_animation', 'intensity_map'):
                print(f'{k:34s} {v}')
        if res['notes']:
            print('\nWARNING:', ' | '.join(res['notes']))
        if p.frames:
            print('\nFire spread from the ignition line (press play, or drag the slider).\nGIF saved to', res['animation'])
            display(fire_player(p.frames))
        display(Image(filename=res['intensity_map'], width=650))
        display(Image(filename=res['map'], width=650))
button.on_click(_run)

display(w.VBox([wind_kmh, wind_from, preset, w.HBox([m1, m10, m100]), w.HBox([mherb, mwoody]),
                temp, rh, ignition, ros_mult, engine, animate, out_name, button, out]))
print('Calibrated wind range: %.0f-%.0f km/h (outside it the result is an extrapolation).' % CALIBRATED_WIND_KMH)

Calibrated wind range: 7-30 km/h (outside it the result is an extrapolation).


In [ ]:
# Same run without widgets: edit the values and run this cell
p = CustomRun(wind_speed_kmh=22, wind_from_deg=225, moisture='summer', temperature_c=18, relative_humidity_pct=63,
              ignition='east', engine='mtt', out_dir='../custom_run_output', make_animation=True)
res, T, ros, fl = run_custom(p)
display(fire_player(p.frames))
res